# Experimentos de la semana · obstáculos

![Mapas sin obstáculos, con 2 obstáculos verdes y con 3 obstáculos de colores, en dos partidas](https://raw.githubusercontent.com/FabriRandon/r2dreamer-cookie/main/notebooks/experimentos-semana-del-25/mapas_obstaculos.png)

*Triángulo rojo: el agente. Círculo azul: el botón. Los obstáculos cambian de casilla en cada partida.*

- **2 obstáculos verdes:** un cuadrado y un triángulo verdes, uno en cada habitación lateral.
- **3 obstáculos de colores:** un cuadrado verde, un triángulo morado y un rombo amarillo en la habitación del botón.
- **`reward_slices`:** en cada batch se repite una secuencia en la que el agente comió una galleta.

Tres experimentos por vista, de 500.000 pasos cada uno:

| | 2 obstáculos verdes | 3 obstáculos de colores |
|---|---|---|
| sin `reward_slices` | ✓ | — |
| con `reward_slices` | ✓ | ✓ |

**Usa CPU y ejecuta todo** (Entorno de ejecución → Ejecutar todo).

## 1. Preparar

Monta Drive, lee las métricas y crea los videos que falten. La primera vez que falta un video tarda unos minutos.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
#@title Leer las métricas
import json
from pathlib import Path

RAIZ = Path("/content/drive/MyDrive/galleta")

# Los tres experimentos de cada vista: (obstáculos, reward_slices) → carpeta en Drive.
RUNS = {
    "5×5": {
        ("verdes", False): "semana24/obstaculos_aleatorios5x5/distinct_s0",
        ("verdes", True): "semana24/obstaculos_aleatorios5x5/distinct_sampler1_s0",
        ("colores", True): "semana25/tres_obstaculos_colores5x5/distinct_sampler1_s0",
    },
    "3×3": {
        ("verdes", False): "semana24/obstaculos_aleatorios3x3/distinct_s0",
        ("verdes", True): "semana24/obstaculos_aleatorios3x3/distinct_sampler1_s0",
        ("colores", True): "semana25/tres_obstaculos_colores3x3/distinct_sampler1_s0",
    },
}
OBSTACULOS = {"verdes": "2 obstáculos verdes", "colores": "3 obstáculos de colores"}
COLORES = {"verdes": "#257de0", "colores": "#f47230"}  # el color dice los obstáculos


def nombre(obstaculos, slices):
    return f"{OBSTACULOS[obstaculos]} · {'con' if slices else 'sin'} reward_slices"


def leer(carpeta):
    """Evaluaciones y galletas por partida de entrenamiento, como listas de (paso, valor)."""
    evaluaciones, partidas = [], []
    with open(RAIZ / carpeta / "metrics.jsonl") as archivo:
        for linea in archivo:
            fila = json.loads(linea)
            for clave, serie in (("episode/eval_score", evaluaciones), ("episode/score", partidas)):
                if clave in fila:
                    # Al retomar se vuelve a un paso anterior: se borra lo que se repite.
                    while serie and serie[-1][0] >= fila["step"]:
                        serie.pop()
                    serie.append((fila["step"], fila[clave]))
    return evaluaciones, partidas


DATOS = {}
for vista, runs in RUNS.items():
    for (obstaculos, slices), carpeta in runs.items():
        try:
            DATOS[vista, obstaculos, slices] = leer(carpeta)
            print(f"{vista} · {nombre(obstaculos, slices)}: listo")
        except FileNotFoundError:
            print(f"{vista} · {nombre(obstaculos, slices)}: NO ENCONTRADO en {carpeta}")

In [ ]:
#@title Crear los videos que falten (CPU)
import os
import shutil
import subprocess
import sys
import tempfile

TRABAJO = Path("/content")
REPO = TRABAJO / "r2dreamer-videos"
URL = "https://github.com/FabriRandon/r2dreamer-cookie.git"
REVISION = "bdffa746f32dac6e6d9c7f37fc8f6138a671e117"
SEMILLA_VIDEO = 0
# Cada semana entrenó con su propia versión del entorno de obstáculos.
ENTORNO = {
    "semana24": "notebooks/experimentos-semana-del-24/cookie.py",
    "semana25": "notebooks/experimentos-semana-del-25/cookie.py",
}


def video_de(carpeta):
    archivo = RAIZ / carpeta / f"agente_episodio{SEMILLA_VIDEO}.mp4"
    try:
        return archivo if archivo.is_file() and archivo.stat().st_size > 0 else None
    except OSError:
        return None  # Drive puede estar sincronizando el archivo


def al_dia(carpeta):
    """El video está al día si se creó después del último checkpoint."""
    archivo, checkpoint = video_de(carpeta), RAIZ / carpeta / "latest.pt"
    try:
        return archivo is not None and archivo.stat().st_mtime >= checkpoint.stat().st_mtime
    except OSError:
        return False


pendientes = []
for vista, runs in RUNS.items():
    for (obstaculos, slices), carpeta in runs.items():
        if not (RAIZ / carpeta / "latest.pt").exists():
            print(f"{vista} · {nombre(obstaculos, slices)}: no hay checkpoint en Drive.")
        elif not al_dia(carpeta):
            pendientes.append((f"{vista} · {nombre(obstaculos, slices)}", carpeta))

if not pendientes:
    print("Los seis videos ya están hechos con el último checkpoint.")
else:
    print("Falta 1 video." if len(pendientes) == 1 else f"Faltan {len(pendientes)} videos.", "Preparando el código…")
    if not (REPO / ".git").is_dir():
        REPO.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "init", "--quiet"], cwd=REPO, check=True)
        subprocess.run(["git", "fetch", "--quiet", "--depth=1", URL, REVISION], cwd=REPO, check=True)
        subprocess.run(["git", "checkout", "--quiet", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
    PYTHON = REPO / ".venv/bin/python"
    if not PYTHON.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        UV = [sys.executable, "-m", "uv"]
        subprocess.run([*UV, "venv", "--quiet", "--python", "3.11", ".venv"], cwd=REPO, check=True)
        (REPO / "cpu.txt").write_text("torch==2.8.0+cpu\n")
        subprocess.run([*UV, "pip", "install", "--quiet", "--python", str(PYTHON),
                        "--extra-index-url", "https://download.pytorch.org/whl/cpu",
                        "--index-strategy", "unsafe-best-match", "--constraint", "cpu.txt",
                        "-e", ".[cookie]"], cwd=REPO, check=True)
    temporal = Path(tempfile.mkdtemp(prefix="videos-", dir=TRABAJO))
    for titulo, carpeta in pendientes:
        print(f"Creando video: {titulo}")
        shutil.copyfile(REPO / ENTORNO[carpeta.split("/")[0]], REPO / "envs/cookie.py")
        provisional = temporal / f"{carpeta.replace('/', '_')}.mp4"
        proceso = subprocess.run([str(PYTHON), "watch.py", "--logdir", str(RAIZ / carpeta),
                                  "--out", str(provisional), "--seed", str(SEMILLA_VIDEO)],
                                 cwd=REPO, env={**os.environ, "CUDA_VISIBLE_DEVICES": ""},
                                 text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if proceso.returncode == 0 and provisional.exists():
            shutil.copyfile(provisional, RAIZ / carpeta / f"agente_episodio{SEMILLA_VIDEO}.mp4")
            print("  ", proceso.stdout.strip().splitlines()[-1].split(" Video en ")[0])
        else:
            print("   No se pudo crear. Vuelve a ejecutar esta celda.")
            print("\n".join(proceso.stdout.splitlines()[-10:]))

In [ ]:
#@title Funciones del gráfico y de los videos
import html

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, Video, display
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter

plt.rcParams.update({
    "font.size": 11, "axes.titlesize": 12, "axes.titlelocation": "left", "axes.titlepad": 10,
    "axes.labelcolor": "#555555", "axes.edgecolor": "#999999",
    "xtick.color": "#555555", "ytick.color": "#555555",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e6e6e6", "axes.axisbelow": True,
    "legend.frameon": False,
})
FIN = 500_000
MAXIMO = 23
TEXTO = "#222222"


def trazo(slices):
    return "-" if slices else "--"  # el trazo dice si hay reward_slices


def rotular_finales(figura, eje, finales, **estilo):
    """Escribe el valor final junto al último punto y separa los que quedarían encima."""
    figura.canvas.draw()
    a_puntos = 72 / figura.dpi
    separacion = 1.3 * plt.rcParams["font.size"]
    alturas = sorted((eje.transData.transform((x, y))[1] * a_puntos, x, y, texto) for x, y, texto in finales)
    anterior = None
    for altura, x, y, texto in alturas:
        destino = altura if anterior is None else max(altura, anterior + separacion)
        eje.annotate(texto, xy=(x, y), xytext=(8, destino - altura), textcoords="offset points",
                     va="center", color=TEXTO, **estilo)
        anterior = destino


def grafico(vista):
    figura, (izquierda, derecha) = plt.subplots(1, 2, figsize=(11, 3.6), layout="constrained")
    figura.suptitle(f"Vista {vista}", x=0.01, ha="left", fontsize=14, fontweight="bold")
    for eje in (izquierda, derecha):
        eje.set_xlim(-10_000, 580_000)
        eje.set_xticks(range(0, FIN + 1, 100_000))
        eje.xaxis.set_major_formatter(FuncFormatter(lambda x, _: "0" if x == 0 else f"{x / 1000:.0f}k"))
        eje.set_xlabel("Pasos de entrenamiento")

    izquierda.set_title("Galletas comidas al juntar datos (escala log)")
    izquierda.set_yscale("log", nonpositive="clip")
    izquierda.set_ylim(0.8, 14_000)
    izquierda.yaxis.set_major_locator(LogLocator(base=10))
    izquierda.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:,.0f}".replace(",", ".")))
    izquierda.yaxis.set_minor_formatter(NullFormatter())

    derecha.set_title("Evaluación: galletas por partida")
    derecha.set_ylim(-0.8, 25)
    derecha.set_yticks(range(0, 26, 5))
    derecha.axhline(MAXIMO, color="#777777", linestyle=":", linewidth=1.2)
    derecha.annotate(f"máximo ≈ {MAXIMO}", xy=(0, MAXIMO), xycoords=("axes fraction", "data"),
                     xytext=(4, 3), textcoords="offset points", va="bottom", color="#777777")

    galletas, evaluacion, leyenda = [], [], []
    for (obstaculos, slices) in RUNS[vista]:
        if (vista, obstaculos, slices) not in DATOS:
            continue
        evaluaciones, partidas = DATOS[vista, obstaculos, slices]
        color, estilo = COLORES[obstaculos], trazo(slices)
        pasos = np.array([0] + [paso for paso, _ in partidas])
        total = np.cumsum([0] + [valor for _, valor in partidas])
        izquierda.step(pasos, total, where="post", color=color, linestyle=estilo, linewidth=2)
        izquierda.plot(pasos[-1], total[-1], "o", color=color, markersize=6)
        galletas.append((pasos[-1], total[-1], f"{total[-1]:,.0f}".replace(",", ".")))
        x, y = np.array(evaluaciones).T
        derecha.plot(x, y, color=color, linestyle=estilo, linewidth=2)
        derecha.plot(x[-1], y[-1], "o", color=color, markersize=6)
        evaluacion.append((x[-1], y[-1], f"{y[-1]:.1f}".replace(".", ",")))
        leyenda.append((Line2D([], [], color=color, linestyle=estilo, linewidth=2), nombre(obstaculos, slices)))
    rotular_finales(figura, izquierda, galletas, fontweight="bold")
    rotular_finales(figura, derecha, evaluacion, fontweight="bold")
    figura.legend(*zip(*leyenda), loc="outside lower center", ncol=len(leyenda))
    plt.show()


AMPLIAR = (
    "const t=this.closest('.tarjeta');"
    "const grande=t.classList.toggle('grande');"
    "this.textContent=grande?'Achicar':'Ampliar';"
    "t.scrollIntoView({block:'start',behavior:'smooth'});"
)
ESTILOS = (
    "<style>"
    ".fila{display:grid;grid-template-columns:repeat(auto-fit,minmax(300px,1fr));gap:12px;max-width:1100px}"
    ".tarjeta{border:1px solid #8885;border-radius:8px;padding:10px}"
    ".tarjeta.grande{grid-column:1/-1}"
    ".cabeza{display:flex;justify-content:space-between;align-items:center;gap:8px;margin-bottom:6px}"
    ".cabeza b{font-size:13px}"
    ".cabeza button{font:inherit;padding:2px 10px;border:1px solid #8888;border-radius:6px;"
    "background:transparent;color:inherit;cursor:pointer}"
    ".tarjeta video{width:100%;height:auto;image-rendering:pixelated}"
    "</style>"
)


def tarjeta(vista, obstaculos, slices):
    carpeta = RUNS[vista][obstaculos, slices]
    titulo = html.escape(nombre(obstaculos, slices))
    archivo = video_de(carpeta)
    if archivo is None:
        return f'<div class="tarjeta"><b>{titulo}</b><p>No hay video: ejecuta «Crear los videos que falten».</p></div>'
    try:
        reproductor = Video(str(archivo), embed=True)._repr_html_()
    except OSError:
        return f'<div class="tarjeta"><b>{titulo}</b><p>Drive aún no pudo leer el video. Vuelve a ejecutar esta celda.</p></div>'
    return (f'<div class="tarjeta"><div class="cabeza"><b>{titulo}</b>'
            f'<button onclick="{AMPLIAR}">Ampliar</button></div>{reproductor}</div>')


def videos(vista):
    display(HTML(ESTILOS + '<div class="fila">'
                 + "".join(tarjeta(vista, o, s) for o, s in RUNS[vista]) + "</div>"))

## 2. Vista 5×5

Color: tipo de obstáculo. Línea punteada: sin `reward_slices`. Al final de cada línea, su valor a los 500.000 pasos.

In [ ]:
grafico("5×5")
videos("5×5")

## 3. Vista 3×3

En el 3×3 con obstáculos verdes y `reward_slices`, el último checkpoint guardado es de 370.000 pasos: su video es de ese momento.

In [ ]:
grafico("3×3")
videos("3×3")